### 1.5 Simulação do movimento do Otto no MATLAB

Com o modelo cinemático da seção anterior, é possível animar o Otto computacionalmente antes mesmo de ligá-lo fisicamente. A simulação a seguir representa o robô de forma **simplificada**: dois segmentos de perna (esquerda e direita), a plataforma de apoio dos pés e a cabeça — sem braços e sem modelagem dinâmica de forças.

**Princípio de funcionamento:**

- Os **servos do quadril** (S1, S2) oscilam em oposição de fase → cada perna inclina lateralmente de forma alternada, transferindo o peso.
- Os **servos dos pés** (S3, S4) oscilam também em oposição de fase, desfasados 90° do quadril → um pé levanta enquanto o outro apoia, criando o passo.
- A **cabeça** permanece rígida sobre o corpo (sem servo próprio nesta simulação).



In [28]:
from IPython.display import HTML, display
import uuid

cid = f"otto_{uuid.uuid4().hex[:8]}"

html = f"""
<div style="
font-family:sans-serif;
border:1px solid #ccc;
border-radius:8px;
padding:14px;
max-width:920px;
background:#fafafa;
">

<div style="
font-size:0.9rem;
font-weight:700;
letter-spacing:.05em;
text-transform:uppercase;
margin-bottom:10px;
color:#444;
">
Simulador — Otto DIY Cinemática
</div>

<canvas id="{cid}" width="860" height="560"
style="
display:block;
background:white;
border:1px solid #ddd;
border-radius:6px;
"></canvas>

<div style="
margin-top:14px;
display:grid;
grid-template-columns:repeat(3,1fr);
gap:14px;
font-family:monospace;
font-size:12px;
">

<div>
Amplitude quadril<br>
<input type="range" id="{cid}_hip" min="0" max="35" value="16">
<span id="{cid}_hipv">16°</span>
</div>

<div>
Amplitude pé<br>
<input type="range" id="{cid}_foot" min="0" max="35" value="12">
<span id="{cid}_footv">12°</span>
</div>

<div>
Velocidade<br>
<input type="range" id="{cid}_spd" min="1" max="10" value="4">
<span id="{cid}_spdv">4</span>
</div>

<div>
Ângulo base pé esquerdo<br>
<input type="range" id="{cid}_baseL" min="-90" max="0" value="-35">
<span id="{cid}_baseLv">-35°</span>
</div>

<div>
Ângulo base pé direito<br>
<input type="range" id="{cid}_baseR" min="0" max="90" value="35">
<span id="{cid}_baseRv">35°</span>
</div>

<div>
Altura perna<br>
<input type="range" id="{cid}_leg" min="80" max="180" value="120">
<span id="{cid}_legv">120</span>
</div>

</div>
</div>

<script>
(function(){{

const cv = document.getElementById("{cid}");
const ctx = cv.getContext("2d");

let t = 0;

function rad(v){{
    return v*Math.PI/180;
}}

function val(id){{
    return parseFloat(document.getElementById(id).value);
}}

function txt(id,v){{
    document.getElementById(id).textContent=v;
}}

function rr(x,y,w,h,r){{
    ctx.beginPath();
    ctx.roundRect(x,y,w,h,r);
    ctx.fill();
}}

function draw(){{
    
    requestAnimationFrame(draw);

    const W = cv.width;
    const H = cv.height;

    ctx.clearRect(0,0,W,H);

    // sliders

    const HIP_A   = rad(val("{cid}_hip"));
    const FOOT_A  = rad(val("{cid}_foot"));
    const SPD     = val("{cid}_spd") * 0.028;

    const BASE_L  = rad(val("{cid}_baseL"));
    const BASE_R  = rad(val("{cid}_baseR"));

    const LEG_H   = val("{cid}_leg");

    txt("{cid}_hipv", val("{cid}_hip")+"°");
    txt("{cid}_footv", val("{cid}_foot")+"°");
    txt("{cid}_spdv", val("{cid}_spd"));

    txt("{cid}_baseLv", val("{cid}_baseL")+"°");
    txt("{cid}_baseRv", val("{cid}_baseR")+"°");

    txt("{cid}_legv", val("{cid}_leg"));

    t += SPD;

    // movimento

    const hipL = HIP_A * Math.sin(t);
    const hipR = HIP_A * Math.sin(t + Math.PI);

    const footL = FOOT_A * Math.sin(t + Math.PI/2);
    const footR = FOOT_A * Math.sin(t + 3*Math.PI/2);

    // dimensões

    const BODY_W = 150;
    const BODY_H = 150;

    const LEG_W  = 26;

    const FOOT_W = 95;
    const FOOT_H = 20;

    const HIP_DIST = 90;

    const cx = W/2;

    const groundY = H - 50;

    // corpo sobe pelos pés

    const liftL = Math.max(0, Math.sin(footL))*28;
    const liftR = Math.max(0, Math.sin(footR))*28;

    const bodyLift = Math.max(liftL,liftR);

    const bodyTop =
        groundY
        - FOOT_H
        - LEG_H
        - BODY_H
        - bodyLift;

    const hipY = bodyTop + BODY_H;

    const hipLX = cx - HIP_DIST/2;
    const hipRX = cx + HIP_DIST/2;

    // chão

    ctx.strokeStyle="#cfcfcf";
    ctx.lineWidth=2;

    ctx.beginPath();
    ctx.moveTo(40,groundY);
    ctx.lineTo(W-40,groundY);
    ctx.stroke();

    // pernas

    function drawLeg(side, xHip, hipAng, footAng, baseFoot){{

        // deslocamento lateral apenas

        const ankleX = xHip + Math.sin(hipAng)*50;

        const ankleY = hipY + LEG_H;

        // perna

        ctx.save();

        ctx.translate(xHip, hipY);

        ctx.rotate(hipAng * 0.9);

        ctx.fillStyle="#f2c230";

        rr(
            -LEG_W/2,
            0,
            LEG_W,
            LEG_H,
            5
        );

        ctx.restore();

        // pé

        ctx.save();

        // pivô no calcanhar
        ctx.translate(ankleX, ankleY);

        // orientação lateral natural
        ctx.rotate(baseFoot);

        // pitch do pé
        // esquerdo e direito espelhados
        if(side === "R"){{
            ctx.rotate(-footAng);
        }}else{{
            ctx.rotate(footAng);
        }}

        ctx.fillStyle="#1565C0";

        // pivô no calcanhar
        ctx.beginPath();

        ctx.roundRect(
            0,
            -FOOT_H/2,
            FOOT_W,
            FOOT_H,
            5
        );

        ctx.fill();

        ctx.restore();

        // juntas

        ctx.fillStyle="#666";

        ctx.beginPath();
        ctx.arc(xHip, hipY, 5, 0, 2*Math.PI);
        ctx.fill();

        ctx.beginPath();
        ctx.arc(ankleX, ankleY, 4, 0, 2*Math.PI);
        ctx.fill();
    }}

    drawLeg(
        "L",
        hipLX,
        hipL,
        footL,
        BASE_L
    );

    drawLeg(
        "R",
        hipRX,
        hipR,
        footR,
        BASE_R
    );

    // corpo

    ctx.fillStyle="#1565C0";

    rr(
        cx - BODY_W/2,
        bodyTop,
        BODY_W,
        BODY_H,
        12
    );

    // olhos

    const eyeY = bodyTop + 62;

    ctx.fillStyle="#dff6ff";

    ctx.beginPath();
    ctx.arc(cx-32, eyeY, 22, 0, 2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(cx+32, eyeY, 22, 0, 2*Math.PI);
    ctx.fill();

    ctx.fillStyle="#111";

    ctx.beginPath();
    ctx.arc(cx-32, eyeY, 7, 0, 2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(cx+32, eyeY, 7, 0, 2*Math.PI);
    ctx.fill();

}}

draw();

}})();
</script>
"""

display(HTML(html))

In [29]:
from IPython.display import HTML, display
import uuid

cid = f"otto_{uuid.uuid4().hex[:8]}"

html = f"""
<div style="
font-family:sans-serif;
border:1px solid #ccc;
border-radius:8px;
padding:14px;
max-width:920px;
background:#fafafa;
">

<div style="
font-size:0.9rem;
font-weight:700;
letter-spacing:.05em;
text-transform:uppercase;
margin-bottom:10px;
color:#444;
">
Simulador — Otto DIY Cinemática
</div>

<canvas id="{cid}" width="860" height="560"
style="
display:block;
background:white;
border:1px solid #ddd;
border-radius:6px;
"></canvas>

<div style="
margin-top:14px;
display:grid;
grid-template-columns:repeat(3,1fr);
gap:14px;
font-family:monospace;
font-size:12px;
">

<div>
Amplitude quadril<br>
<input type="range" id="{cid}_hip" min="0" max="35" value="16">
<span id="{cid}_hipv">16°</span>
</div>

<div>
Amplitude pé<br>
<input type="range" id="{cid}_foot" min="0" max="35" value="12">
<span id="{cid}_footv">12°</span>
</div>

<div>
Velocidade<br>
<input type="range" id="{cid}_spd" min="1" max="10" value="4">
<span id="{cid}_spdv">4</span>
</div>

<div>
Ângulo base pé esquerdo<br>
<input type="range" id="{cid}_baseL" min="-90" max="90" value="0">
<span id="{cid}_baseLv">0°</span>
</div>

<div>
Ângulo base pé direito<br>
<input type="range" id="{cid}_baseR" min="-90" max="90" value="0">
<span id="{cid}_baseRv">0°</span>
</div>

<div>
Altura perna<br>
<input type="range" id="{cid}_leg" min="80" max="180" value="120">
<span id="{cid}_legv">120</span>
</div>

</div>
</div>

<script>
(function(){{

const cv = document.getElementById("{cid}");
const ctx = cv.getContext("2d");

let t = 0;

function rad(v){{
    return v*Math.PI/180;
}}

function val(id){{
    return parseFloat(document.getElementById(id).value);
}}

function txt(id,v){{
    document.getElementById(id).textContent=v;
}}

function rr(x,y,w,h,r){{
    ctx.beginPath();
    ctx.roundRect(x,y,w,h,r);
    ctx.fill();
}}

function draw(){{
    
    requestAnimationFrame(draw);

    const W = cv.width;
    const H = cv.height;

    ctx.clearRect(0,0,W,H);

    // sliders

    const HIP_A   = rad(val("{cid}_hip"));
    const FOOT_A  = rad(val("{cid}_foot"));
    const SPD     = val("{cid}_spd") * 0.028;

    const BASE_L  = rad(val("{cid}_baseL"));
    const BASE_R  = rad(val("{cid}_baseR"));

    const LEG_H   = val("{cid}_leg");

    txt("{cid}_hipv", val("{cid}_hip")+"°");
    txt("{cid}_footv", val("{cid}_foot")+"°");
    txt("{cid}_spdv", val("{cid}_spd"));

    txt("{cid}_baseLv", val("{cid}_baseL")+"°");
    txt("{cid}_baseRv", val("{cid}_baseR")+"°");

    txt("{cid}_legv", val("{cid}_leg"));

    t += SPD;

    // movimentos

    const hipL = HIP_A * Math.sin(t);
    const hipR = HIP_A * Math.sin(t + Math.PI);

    const footL = FOOT_A * Math.sin(t + Math.PI/2);
    const footR = FOOT_A * Math.sin(t + 3*Math.PI/2);

    // dimensões

    const BODY_W = 150;
    const BODY_H = 150;

    const LEG_W  = 26;

    const FOOT_W = 95;
    const FOOT_H = 20;

    const HIP_DIST = 90;

    const cx = W/2;

    const groundY = H - 50;

    // corpo sobe pelos pés

    const liftL = Math.max(0, Math.sin(footL))*28;
    const liftR = Math.max(0, Math.sin(footR))*28;

    const bodyLift = Math.max(liftL,liftR);

    const bodyTop =
        groundY
        - FOOT_H
        - LEG_H
        - BODY_H
        - bodyLift;

    const hipY = bodyTop + BODY_H;

    const hipLX = cx - HIP_DIST/2;
    const hipRX = cx + HIP_DIST/2;

    // chão

    ctx.strokeStyle="#cfcfcf";
    ctx.lineWidth=2;

    ctx.beginPath();
    ctx.moveTo(40,groundY);
    ctx.lineTo(W-40,groundY);
    ctx.stroke();

    // pernas

    function drawLeg(side, xHip, hipAng, footAng, baseFoot){{

        // quadril gira no plano XY
        const ankleX = xHip + Math.sin(hipAng)*50;

        const ankleY = hipY + LEG_H;

        // perna

        ctx.save();

        ctx.translate(xHip, hipY);

        ctx.rotate(hipAng * 0.9);

        ctx.fillStyle="#f2c230";

        rr(
            -LEG_W/2,
            0,
            LEG_W,
            LEG_H,
            5
        );

        ctx.restore();

        // pé

        ctx.save();

        // pivô no calcanhar
        ctx.translate(ankleX, ankleY);

        // 0 = piso
        // +90 = ponta sobe
        // -90 = ponta desce

        let finalAng;

        if(side === "L"){{
            // pé esquerdo aponta para esquerda
            finalAng = Math.PI + baseFoot + footAng;
        }}else{{
            // pé direito aponta para direita
            finalAng = baseFoot - footAng;
        }}

        ctx.rotate(finalAng);

        ctx.fillStyle="#1565C0";

        // pé desenhado a partir do calcanhar
        ctx.beginPath();

        ctx.roundRect(
            0,
            -FOOT_H/2,
            FOOT_W,
            FOOT_H,
            5
        );

        ctx.fill();

        ctx.restore();

        // juntas

        ctx.fillStyle="#666";

        ctx.beginPath();
        ctx.arc(xHip, hipY, 5, 0, 2*Math.PI);
        ctx.fill();

        ctx.beginPath();
        ctx.arc(ankleX, ankleY, 4, 0, 2*Math.PI);
        ctx.fill();
    }}

    drawLeg(
        "L",
        hipLX,
        hipL,
        footL,
        BASE_L
    );

    drawLeg(
        "R",
        hipRX,
        hipR,
        footR,
        BASE_R
    );

    // corpo

    ctx.fillStyle="#1565C0";

    rr(
        cx - BODY_W/2,
        bodyTop,
        BODY_W,
        BODY_H,
        12
    );

    // olhos

    const eyeY = bodyTop + 62;

    ctx.fillStyle="#dff6ff";

    ctx.beginPath();
    ctx.arc(cx-32, eyeY, 22, 0, 2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(cx+32, eyeY, 22, 0, 2*Math.PI);
    ctx.fill();

    ctx.fillStyle="#111";

    ctx.beginPath();
    ctx.arc(cx-32, eyeY, 7, 0, 2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(cx+32, eyeY, 7, 0, 2*Math.PI);
    ctx.fill();

}}

draw();

}})();
</script>
"""

display(HTML(html))

In [30]:
from IPython.display import HTML, display
import uuid

cid = f"otto_{uuid.uuid4().hex[:8]}"

html = f"""
<div style="
font-family:sans-serif;
border:1px solid #ccc;
border-radius:8px;
padding:14px;
max-width:920px;
background:#fafafa;
">

<div style="
font-size:0.9rem;
font-weight:700;
letter-spacing:.05em;
text-transform:uppercase;
margin-bottom:10px;
color:#444;
">
Simulador — Otto DIY Cinemática
</div>

<canvas id="{cid}" width="860" height="560"
style="
display:block;
background:white;
border:1px solid #ddd;
border-radius:6px;
"></canvas>

<div style="
margin-top:14px;
display:grid;
grid-template-columns:repeat(3,1fr);
gap:14px;
font-family:monospace;
font-size:12px;
">

<div>
Amplitude quadril<br>
<input type="range" id="{cid}_hip" min="0" max="35" value="16">
<span id="{cid}_hipv">16°</span>
</div>

<div>
Amplitude pé<br>
<input type="range" id="{cid}_foot" min="0" max="35" value="12">
<span id="{cid}_footv">12°</span>
</div>

<div>
Velocidade<br>
<input type="range" id="{cid}_spd" min="1" max="10" value="4">
<span id="{cid}_spdv">4</span>
</div>

<div>
Ângulo base pé esquerdo<br>
<input type="range" id="{cid}_baseL" min="-90" max="90" value="0">
<span id="{cid}_baseLv">0°</span>
</div>

<div>
Ângulo base pé direito<br>
<input type="range" id="{cid}_baseR" min="-90" max="90" value="0">
<span id="{cid}_baseRv">0°</span>
</div>

<div>
Altura perna<br>
<input type="range" id="{cid}_leg" min="80" max="180" value="120">
<span id="{cid}_legv">120</span>
</div>

</div>
</div>

<script>
(function(){{

const cv = document.getElementById("{cid}");
const ctx = cv.getContext("2d");

let t = 0;

function rad(v){{
    return v*Math.PI/180;
}}

function val(id){{
    return parseFloat(document.getElementById(id).value);
}}

function txt(id,v){{
    document.getElementById(id).textContent=v;
}}

function rr(x,y,w,h,r){{
    ctx.beginPath();
    ctx.roundRect(x,y,w,h,r);
    ctx.fill();
}}

function draw(){{
    
    requestAnimationFrame(draw);

    const W = cv.width;
    const H = cv.height;

    ctx.clearRect(0,0,W,H);

    // sliders

    const HIP_A   = rad(val("{cid}_hip"));
    const FOOT_A  = rad(val("{cid}_foot"));
    const SPD     = val("{cid}_spd") * 0.028;

    const BASE_L  = rad(val("{cid}_baseL"));
    const BASE_R  = rad(val("{cid}_baseR"));

    const LEG_H   = val("{cid}_leg");

    txt("{cid}_hipv", val("{cid}_hip")+"°");
    txt("{cid}_footv", val("{cid}_foot")+"°");
    txt("{cid}_spdv", val("{cid}_spd"));

    txt("{cid}_baseLv", val("{cid}_baseL")+"°");
    txt("{cid}_baseRv", val("{cid}_baseR")+"°");

    txt("{cid}_legv", val("{cid}_leg"));

    t += SPD;

    // movimento do Otto

    const hipL = HIP_A * Math.sin(t);
    const hipR = HIP_A * Math.sin(t + Math.PI);

    const footL = FOOT_A * Math.sin(t + Math.PI/2);
    const footR = FOOT_A * Math.sin(t + 3*Math.PI/2);

    // dimensões

    const BODY_W = 150;
    const BODY_H = 150;

    const LEG_W  = 26;

    const FOOT_W = 95;
    const FOOT_H = 20;

    const HIP_DIST = 90;

    const cx = W/2;

    const groundY = H - 50;

    // corpo sobe pelos pés

    const liftL = Math.max(0, Math.sin(footL))*28;
    const liftR = Math.max(0, Math.sin(footR))*28;

    const bodyLift = Math.max(liftL,liftR);

    const bodyTop =
        groundY
        - FOOT_H
        - LEG_H
        - BODY_H
        - bodyLift;

    const hipY = bodyTop + BODY_H;

    const hipLX = cx - HIP_DIST/2;
    const hipRX = cx + HIP_DIST/2;

    // chão

    ctx.strokeStyle="#cfcfcf";
    ctx.lineWidth=2;

    ctx.beginPath();
    ctx.moveTo(40,groundY);
    ctx.lineTo(W-40,groundY);
    ctx.stroke();

    // pernas

    function drawLeg(side, xHip, hipAng, footAng, baseFoot){{

        // perna rígida girando no plano XY

        const LEG_LEN = LEG_H;

        const rot = hipAng;

        // posição do tornozelo

        const ankleX = xHip + Math.sin(rot) * LEG_LEN;
        const ankleY = hipY + Math.cos(rot) * LEG_LEN;

        // perna retangular rígida

        ctx.save();

        ctx.translate(xHip, hipY);

        ctx.rotate(rot);

        ctx.fillStyle="#f2c230";

        ctx.beginPath();

        ctx.roundRect(
            -LEG_W/2,
            0,
            LEG_W,
            LEG_LEN,
            5
        );

        ctx.fill();

        ctx.restore();

        // pé

        ctx.save();

        // pivô no calcanhar
        ctx.translate(ankleX, ankleY);

        // 0 = piso
        // +90 = ponta sobe
        // -90 = ponta desce

        let finalAng;

        if(side === "L"){{
            // pé esquerdo aponta para esquerda
            finalAng = Math.PI + baseFoot + footAng;
        }}else{{
            // pé direito aponta para direita
            finalAng = baseFoot - footAng;
        }}

        ctx.rotate(finalAng);

        ctx.fillStyle="#1565C0";

        ctx.beginPath();

        ctx.roundRect(
            0,
            -FOOT_H/2,
            FOOT_W,
            FOOT_H,
            5
        );

        ctx.fill();

        ctx.restore();

        // juntas

        ctx.fillStyle="#666";

        ctx.beginPath();
        ctx.arc(xHip, hipY, 5, 0, 2*Math.PI);
        ctx.fill();

        ctx.beginPath();
        ctx.arc(ankleX, ankleY, 4, 0, 2*Math.PI);
        ctx.fill();
    }}

    drawLeg(
        "L",
        hipLX,
        hipL,
        footL,
        BASE_L
    );

    drawLeg(
        "R",
        hipRX,
        hipR,
        footR,
        BASE_R
    );

    // corpo

    ctx.fillStyle="#1565C0";

    rr(
        cx - BODY_W/2,
        bodyTop,
        BODY_W,
        BODY_H,
        12
    );

    // olhos

    const eyeY = bodyTop + 62;

    ctx.fillStyle="#dff6ff";

    ctx.beginPath();
    ctx.arc(cx-32, eyeY, 22, 0, 2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(cx+32, eyeY, 22, 0, 2*Math.PI);
    ctx.fill();

    ctx.fillStyle="#111";

    ctx.beginPath();
    ctx.arc(cx-32, eyeY, 7, 0, 2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(cx+32, eyeY, 7, 0, 2*Math.PI);
    ctx.fill();

}}

draw();

}})();
</script>
"""

display(HTML(html))

In [31]:
from IPython.display import HTML, display
import uuid

cid = f"otto3d_{uuid.uuid4().hex[:8]}"

html = f"""
<div style="
font-family:sans-serif;
border:1px solid #ccc;
border-radius:10px;
padding:14px;
max-width:980px;
background:#fafafa;
">

<div style="
font-size:0.9rem;
font-weight:700;
letter-spacing:.05em;
text-transform:uppercase;
margin-bottom:12px;
color:#444;
">
Otto DIY — Modelo cinemático pseudo-3D
</div>

<canvas id="{cid}" width="920" height="620"
style="
display:block;
background:white;
border:1px solid #ddd;
border-radius:6px;
"></canvas>

<div style="
margin-top:14px;
display:grid;
grid-template-columns:repeat(4,1fr);
gap:12px;
font-family:monospace;
font-size:12px;
">

<div>
Amplitude quadril<br>
<input type="range" id="{cid}_hipA" min="0" max="35" value="18">
<span id="{cid}_hipAv">18°</span>
</div>

<div>
Amplitude pé<br>
<input type="range" id="{cid}_footA" min="0" max="35" value="14">
<span id="{cid}_footAv">14°</span>
</div>

<div>
Velocidade<br>
<input type="range" id="{cid}_spd" min="1" max="10" value="4">
<span id="{cid}_spdv">4</span>
</div>

<div>
Escala<br>
<input type="range" id="{cid}_sc" min="70" max="170" value="110">
<span id="{cid}_scv">110%</span>
</div>

</div>
</div>

<script>
(function(){{

const cv = document.getElementById("{cid}");
const ctx = cv.getContext("2d");

let t = 0;

function val(id){{
    return parseFloat(document.getElementById(id).value);
}}

function txt(id,v){{
    document.getElementById(id).textContent=v;
}}

function rad(v){{
    return v*Math.PI/180;
}}

// projeção pseudo-3D

function proj(x,y,z){{
    return {{
        x: 460 + x*1.0 + z*0.55,
        y: 520 - y*1.0 - z*0.18
    }};
}}

function box3D(cx,cy,cz,w,h,d,col){{
    
    const p1 = proj(cx-w/2, cy, cz-d/2);
    const p2 = proj(cx+w/2, cy, cz-d/2);
    const p3 = proj(cx+w/2, cy+h, cz-d/2);
    const p4 = proj(cx-w/2, cy+h, cz-d/2);

    const p5 = proj(cx-w/2, cy, cz+d/2);
    const p6 = proj(cx+w/2, cy, cz+d/2);
    const p7 = proj(cx+w/2, cy+h, cz+d/2);
    const p8 = proj(cx-w/2, cy+h, cz+d/2);

    ctx.fillStyle = col;

    // topo

    ctx.beginPath();
    ctx.moveTo(p4.x,p4.y);
    ctx.lineTo(p3.x,p3.y);
    ctx.lineTo(p7.x,p7.y);
    ctx.lineTo(p8.x,p8.y);
    ctx.closePath();
    ctx.fill();

    // lateral

    ctx.beginPath();
    ctx.moveTo(p2.x,p2.y);
    ctx.lineTo(p6.x,p6.y);
    ctx.lineTo(p7.x,p7.y);
    ctx.lineTo(p3.x,p3.y);
    ctx.closePath();
    ctx.fill();

    // frente

    ctx.beginPath();
    ctx.moveTo(p1.x,p1.y);
    ctx.lineTo(p2.x,p2.y);
    ctx.lineTo(p3.x,p3.y);
    ctx.lineTo(p4.x,p4.y);
    ctx.closePath();
    ctx.fill();
}}

function draw(){{
    
    requestAnimationFrame(draw);

    const W = cv.width;
    const H = cv.height;

    ctx.clearRect(0,0,W,H);

    // sliders

    const hipA  = rad(val("{cid}_hipA"));
    const footA = rad(val("{cid}_footA"));
    const spd   = val("{cid}_spd")*0.025;
    const scl   = val("{cid}_sc")/100;

    txt("{cid}_hipAv", val("{cid}_hipA")+"°");
    txt("{cid}_footAv", val("{cid}_footA")+"°");
    txt("{cid}_spdv", val("{cid}_spd"));
    txt("{cid}_scv", val("{cid}_sc")+"%");

    t += spd;

    // cinemática correta do Otto

    // quadris:
    // rotação horizontal (yaw)

    const hipL = hipA*Math.sin(t);
    const hipR = hipA*Math.sin(t+Math.PI);

    // pés:
    // rotação vertical (pitch)

    const footL = footA*Math.sin(t+Math.PI/2);
    const footR = footA*Math.sin(t+3*Math.PI/2);

    // dimensões

    const BODY_W = 110*scl;
    const BODY_H = 120*scl;
    const BODY_D = 70*scl;

    const LEG_H = 95*scl;
    const LEG_W = 18*scl;
    const LEG_D = 16*scl;

    const FOOT_W = 75*scl;
    const FOOT_H = 14*scl;
    const FOOT_D = 38*scl;

    const HIP_SPACE = 70*scl;

    // corpo sobe pelos pés

    const liftL = Math.max(0,Math.sin(footL))*30*scl;
    const liftR = Math.max(0,Math.sin(footR))*30*scl;

    const bodyY =
        180 + Math.max(liftL,liftR);

    // chão

    ctx.strokeStyle="#ddd";

    for(let i=-300;i<=300;i+=40){{
        
        let a = proj(i,0,-260);
        let b = proj(i,0,260);

        ctx.beginPath();
        ctx.moveTo(a.x,a.y);
        ctx.lineTo(b.x,b.y);
        ctx.stroke();
    }}

    for(let i=-260;i<=260;i+=40){{
        
        let a = proj(-300,0,i);
        let b = proj(300,0,i);

        ctx.beginPath();
        ctx.moveTo(a.x,a.y);
        ctx.lineTo(b.x,b.y);
        ctx.stroke();
    }}

    // quadris

    const hipLX = -HIP_SPACE/2;
    const hipRX = +HIP_SPACE/2;

    // pernas
    // IMPORTANTÍSSIMO:
    // elas ficam verticais
    // apenas giram horizontalmente no XY

    function leg(side, hipX, hipAng, footAng){{
        
        // rotação lateral da perna
        // movimento em XY

        const ankleX =
            hipX + Math.sin(hipAng)*42*scl;

        const ankleZ =
            Math.cos(hipAng)*18*scl;

        // perna vertical rígida

        box3D(
            ankleX,
            bodyY-LEG_H,
            ankleZ,
            LEG_W,
            LEG_H,
            LEG_D,
            "#f2c230"
        );

        // pé

        const footPitch =
            footAng;

        // pivô no calcanhar

        const heelX = ankleX;
        const heelY = bodyY-LEG_H;
        const heelZ = ankleZ;

        // ponta do pé sobe/desce

        const toeY =
            heelY + Math.sin(footPitch)*20*scl;

        const toeZ =
            heelZ + Math.cos(footPitch)*18*scl;

        // orientação lateral correta

        let footOffset;

        if(side==="L"){{
            footOffset = -FOOT_W/2;
        }}else{{
            footOffset = 0;
        }}

        box3D(
            heelX + footOffset,
            heelY-FOOT_H,
            toeZ,
            FOOT_W,
            FOOT_H,
            FOOT_D,
            "#1565C0"
        );
    }}

    leg("L",hipLX,hipL,footL);
    leg("R",hipRX,hipR,footR);

    // corpo

    box3D(
        0,
        bodyY,
        0,
        BODY_W,
        BODY_H,
        BODY_D,
        "#1565C0"
    );

    // olhos maiores

    let e1 = proj(-26, bodyY+78*scl, -36);
    let e2 = proj( 26, bodyY+78*scl, -36);

    ctx.fillStyle="#dff6ff";

    ctx.beginPath();
    ctx.arc(e1.x,e1.y,18*scl,0,2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(e2.x,e2.y,18*scl,0,2*Math.PI);
    ctx.fill();

    ctx.fillStyle="#111";

    ctx.beginPath();
    ctx.arc(e1.x,e1.y,5*scl,0,2*Math.PI);
    ctx.fill();

    ctx.beginPath();
    ctx.arc(e2.x,e2.y,5*scl,0,2*Math.PI);
    ctx.fill();

}}

draw();

}})();
</script>
"""

display(HTML(html))